In [86]:
# !pip install kagglehub

In [87]:
import kagglehub
import pandas as pd
from sklearn.preprocessing import StandardScaler
import numpy as np
import torch
import torch.nn as nn
from torch.utils.data import TensorDataset
from torch.utils.data import DataLoader

In [88]:
# Download latest version
path = kagglehub.dataset_download("raminhuseyn/airline-customer-satisfaction")
# print(path)

In [89]:
df = pd.read_csv(path + "/Airline_customer_satisfaction.csv")
# df.head()

In [90]:
device = 'cuda' if torch.cuda.is_available() else 'cpu'

In [91]:
### DATA PRE-PROCESSING

# 1. See the shape (Rows, Columns)
print(df.shape)

# 2. See which columns have holes (Missing data)
# print(df.isnull().sum())

# 3. Drop null value rows
df = df.dropna()

# Look, if data is biased toward one value? see percentages:
# print(df['satisfaction'].value_counts(normalize=True))

# Convert target to 0 and 1 
df['satisfaction'] = df['satisfaction'].map({'dissatisfied': 0, 'satisfied': 1})
# print(df['satisfaction'].value_counts())

# We have categorical data, for example, business class, eco and eco plus class. The model doesn't understand the difference and even we assign them number like 1,2,3 the model will think 3 is larger than and more important, so we'll split it into different cols. 

# For categorical data with two possible values, we'll convert them to binary and it does not matter, which one will get which one cause nn just need to know the difference. 

categorical_cols = ['Customer Type','Type of Travel','Class']
df = pd.get_dummies(df,columns=categorical_cols,drop_first=True,dtype=int) # drop first will drop the first value alphabatically from each col, for col with three vals, there will be two cols, if both 0, then it means the third class was true. 

# in the dataset we've numerical cols with varying scale, such as between 0-80 while flight distances are large, the model will give more preferences to flight distances, thinking it as more important. 

# we need to standarize them
# Z-score = x - mean / std.dev
scaler = StandardScaler()
num_cols = ['Age','Flight Distance','Departure Delay in Minutes','Arrival Delay in Minutes']
df[num_cols] = scaler.fit_transform(df[num_cols])

df.head()

(129880, 22)


,satisfaction,Age,Flight Distance,Seat comfort,Departure/Arrival time convenient,Food and drink,Gate location,Inflight wifi service,Inflight entertainment,Online support,...,Baggage handling,Checkin service,Cleanliness,Online boarding,Departure Delay in Minutes,Arrival Delay in Minutes,Customer Type_disloyal Customer,Type of Travel_Personal Travel,Class_Eco,Class_Eco Plus
0,1,1.691495,-1.671090,0,0,0,2,2,4,2,...,3,5,3,2,-0.386036,-0.392329,0,1,1,0
1,1,0.500825,0.470348,0,0,0,3,0,2,2,...,4,2,3,2,7.786328,7.536854,0,1,0,0
2,1,-1.615922,0.152882,0,0,0,3,2,0,2,...,4,4,4,2,-0.386036,-0.392329,0,1,1,0
3,1,1.360753,-1.322461,0,0,0,3,3,4,3,...,1,4,1,3,-0.386036,-0.392329,0,1,1,0
4,1,2.022237,-1.584420,0,0,0,3,4,3,4,...,2,4,2,5,-0.386036,-0.392329,0,1,1,0


In [92]:
# Shuffle and split data
np.random.seed(100)
indices = np.arange(len(df))
np.random.shuffle(indices)

train_split = int(0.8 * len(df))
val_split = int(0.9 * len(df))

# Acts as ticket numbers
train_idx = indices[:train_split]
val_idx = indices[train_split:val_split]
test_idx = indices[val_split:]

# 1. Define X (features) and Y (target)
X = df.drop(columns=['satisfaction']).to_numpy()
# print(df.head())
Y = df['satisfaction'].to_numpy()
# print(Y)

# 2. Extract specific sets 
x_train, y_train = X[train_idx], Y[train_idx]
x_val, y_val = X[val_idx], Y[val_idx]
x_test, y_test = X[test_idx], Y[test_idx]

# print(X.shape)

num_features = x_train.shape[1]

x_train_tensor = torch.as_tensor(x_train,dtype=torch.float).to(device).reshape(-1,num_features)
y_train_tensor = torch.as_tensor(y_train,dtype=torch.float).to(device).reshape(-1,1)
x_val_tensor = torch.as_tensor(x_val,dtype=torch.float).to(device).reshape(-1,num_features)
y_val_tensor = torch.as_tensor(y_val,dtype=torch.float).to(device).reshape(-1,1)  # pytorch linear layer expects (batch_size, features)

# Create the training and validation datasets
train_ds = TensorDataset(x_train_tensor, y_train_tensor) # Pairs up input and output
val_ds = TensorDataset(x_val_tensor, y_val_tensor)


# DataLoader act as conveyer belt, instead of feeding all rows at once, it sends data in batches, which helps model help update weights multiple times (SGD), also this speeds up the math/ease up the math/calculations
train_loader = DataLoader(train_ds, batch_size=64, shuffle=True)
val_loader = DataLoader(val_ds, batch_size=64, shuffle=False) # Don't need to shuffle for validation, shuffling needed only for training. 

print("X shape", X.shape)
print("x-train shape", x_train.shape)
print("x_train_tensor shape", x_train_tensor.shape)


X shape (129487, 22)
x-train shape (103589, 22)
x_train_tensor shape torch.Size([103589, 22])


In [93]:
class AirlineModel(nn.Module):
    # Define the model architecture
    def __init__(self, input_size):
        super().__init__()
        self.hidden1 = nn.Linear(input_size,16) # input to 16 neurons in first layer
        self.hidden2 = nn.Linear(16,8) # 16 inputs from first layer and 8 neurons in 2nd layer
        self.output = nn.Linear(8,1) # 8 inputs from layer 2 and 1 output. 

        # Total parameters
        # w = input_size * out_features/neurons, e.g first layer if input is 16, weights for layer 1 are 16 x 16
        # b = one bias for each neuron, e.g for first layer 16 biases.

        # Activations
        self.ReLU =  nn.ReLU()
        self.tanH = nn.Tanh()
        self.sigmoid = nn.Sigmoid()

    # math flow in forward
    def forward(self, x):
        x = self.hidden1(x)
        x = self.ReLU(x)

        x = self.hidden2(x)
        x = self.tanH(x)

        x = self.output(x)
        x = self.sigmoid(x)

        return x


In [ ]:
# initialize model 
model = AirlineModel(num_features).to(device)            